# ECE334 Lab 1 — Basic SPICE, the interactive way

In this lab you **build circuits in XSchem**, press a button to simulate, and then
**analyse the results here in Python** with `ece334lib`.

### The naming contract
Testbenches and this notebook talk through **net names**. Keep them:

| Net | Meaning |
|-----|---------|
| `in` / `out` | input / output of a logic circuit |
| `g` / `s` | gate-drain / source-bulk of a device under characterization |
| `vdd` / `vss` | 1.8 V supply / ground |

Each testbench button writes a named `.raw` file; the cells below load those files.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ece334lib import sim, measure, plot

VDD = 1.8
XS = "xschem"
%matplotlib inline

## 1. RC step response

Open `xschem/rc_tb.sch`, press **Netlist & Simulate**, then run this cell. The output
charges with $\tau = RC$; with $R=1\,\mathrm{k}\Omega$, $C=1\,\mathrm{pF}$,
$\tau=1\,\mathrm{ns}$ and $t_r=\tau\ln 9\approx2.20\,\mathrm{ns}$.

In [ ]:
rc = sim.netlist_and_run(f"{XS}/rc_tb.sch", output="rc_tb.raw", fmt="raw")
t_rise, _ = measure.edges_10_90(rc, "out", vdd=VDD)
tau = rc.x[np.argmax(rc["out"] >= 0.632 * VDD)]
print(f"tau (63.2% point) = {tau*1e9:.3f} ns   (R*C = 1.000 ns)")
print(f"t_rise (10-90%)   = {t_rise*1e9:.3f} ns   (tau*ln9 = {np.log(9):.3f} ns)")
plot.transient(rc, ["in", "out"])
plt.title("RC step response"); plt.show()

## 2. Device extraction — $K_P$ and $V_t$

Open `xschem/diode_tb.sch`, build a diode-connected NMOS in the DUT (gate+drain → `g`,
source+body → `s`; size $W=10$, $L=2$ — unitless microns), and press **Netlist &
Simulate**. A diode-connected device is always in saturation, so $\sqrt{I_D}$ is linear
in $V_{GS}$: the slope gives $K_P$ and the x-intercept gives $V_t$.

In [ ]:
diode = sim.netlist_and_run(f"{XS}/diode_tb.sch", output="diode_nmos.raw", fmt="raw")
vg, idd = diode["g"], diode["id"]
res = measure.extract_square_law(vg, idd, wl=10 / 2, vmin=1.0, vmax=1.7)
print(f"Vtn = {res['Vt']:.3f} V    KPn = {res['KP']*1e6:.1f} uA/V^2")

fig, ax = plt.subplots()
ax.plot(vg, np.sqrt(np.clip(idd, 0, None)) * 1e3, label="$\\sqrt{I_D}$")
vfit = np.array([res["Vt"], 1.8])
ax.plot(vfit, (res["slope"] * vfit + res["intercept"]) * 1e3, "r--", label="sat. fit")
ax.set_xlabel("$V_{GS}$ (V)"); ax.set_ylabel("$\\sqrt{I_D}$ (mA$^{1/2}$)")
ax.legend(); ax.grid(True, alpha=0.3); plt.show()

Repeat for the PMOS (`pmos_diode.spice`, or your own `diode_tb` with a pfet) to get
$|V_{tp}|$ and $K_{Pp}$. Record all four — you'll use them in §4.

## 3. CMOS inverter — transfer curve & noise margins

Build your inverter in the DUT of `xschem/inv_tb.sch` ($W_n=1$, $W_p=3$, $L=0.5$ —
unitless), press **Netlist & Simulate** (writes `inv_tb_vtc.raw` and `inv_tb_tran.raw`).

In [ ]:
sim.netlist_and_run(f"{XS}/inv_tb.sch", output="inv_tb_tran.raw", fmt="raw")
vtc = sim.run(f"{XS}/inv_tb_vtc.raw")
nm = measure.noise_margins(vtc, "in", "out")
print(f"V_M = {nm['VM']:.3f} V   NM_H = {nm['NMH']:.3f} V   NM_L = {nm['NML']:.3f} V")
plot.vtc(vtc, "in", "out")
plt.title("Inverter voltage transfer curve"); plt.show()

## 4. Inverter transient — propagation delay & edge rates

Measure the 50 %→50 % propagation delay and 10–90 % rise/fall, and compare to the
equivalent-resistance hand estimate built from **your extracted** $K_P$, $V_t$.

In [ ]:
tran = sim.run(f"{XS}/inv_tb_tran.raw")
tpd = measure.prop_delay(tran, "in", "out", vdd=VDD)
t_rise, t_fall = measure.edges_10_90(tran, "out", vdd=VDD)

# hand estimate — plug in YOUR extracted values from section 2
KPn, KPp = 170e-6, 40e-6
Vtn, Vtp = 0.46, 0.50
WLn, WLp = 1.0 / 0.5, 3.0 / 0.5
CL = 0.2e-12
t_fall_hand = 2.2 * (VDD / (KPn * WLn * (VDD - Vtn))) * CL
t_rise_hand = 2.2 * (VDD / (KPp * WLp * (VDD - Vtp))) * CL

table = pd.DataFrame({
    "metric":    ["t_rise (ns)", "t_fall (ns)", "t_pd (ns)"],
    "simulated": [t_rise*1e9, t_fall*1e9, tpd*1e9],
    "hand est.": [t_rise_hand*1e9, t_fall_hand*1e9, np.nan],
})
display(table.round(3))
plot.transient(tran, ["in", "out"])
plt.title("Inverter transient response"); plt.show()

### Deliverables
- [ ] RC: measured $\tau$ and $t_r$ vs $RC$
- [ ] Extraction: $V_{tn}$, $K_{Pn}$, $|V_{tp}|$, $K_{Pp}$ with fit plots
- [ ] Inverter VTC: $V_M$ and noise margins
- [ ] Inverter transient: simulated vs hand-estimated $t_r$, $t_f$, $t_{pd}$

Sweeping a parameter is one line with `sim.sweep(...)` — see Lab 2.